In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df1=pd.read_csv('/content/movies.csv')
df1.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [ ]:
df2=pd.read_csv('/content/ratings.csv')
df2.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [ ]:
df1.duplicated().sum()

np.int64(0)

In [ ]:
df2.duplicated().sum()

np.int64(0)

In [ ]:
df2['movieId'].value_counts()

,count
movieId,
356,329
318,317
296,307
593,279
2571,278
...,...
188833,1
189381,1
3899,1


In [ ]:
df=pd.merge(df1,df2,on='movieId')
df.head()

,movieId,title,genres,userId,rating,timestamp
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1,4.0,964982703
1,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,5,4.0,847434962
2,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,7,4.5,1106635946
3,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,15,2.5,1510577970
4,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,17,4.5,1305696483


In [ ]:
print(df1.shape)
print(df2.shape)
print(df.shape)

(9742, 3)
(100836, 4)
(100836, 6)


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
import re

In [ ]:
df['title']=df['title'].apply(lambda x:re.sub(r"\s*\(\d{4}\)",'',x))

In [ ]:
df['title']

,title
0,Toy Story
1,Toy Story
2,Toy Story
3,Toy Story
4,Toy Story
...,...
100831,Black Butler: Book of the Atlantic
100832,No Game No Life: Zero
100833,Flint
100834,Bungo Stray Dogs: Dead Apple


df1 has unique movie_ids (probably, since it’s movie metadata).

df2 has many ratings per movie (many users rate the same movie).

| movie\_id | title   | genre  | userId | rating | time       |
| --------- | ------- | ------ | ------ | ------ | ---------- |
| 1         | Movie A | Action | 101    | 5      | 2023-01-01 |
| 1         | Movie A | Action | 102    | 4      | 2023-01-02 |
| 2         | Movie B | Comedy | 103    | 3      | 2023-01-01 |
| 3         | Movie C | Drama  | 104    | 4      | 2023-01-03 |
| ...       | ...     | ...    | ...    | ...    | ...        |


In [ ]:
from scipy.sparse import coo_matrix


In [ ]:
user_map=df['userId'].astype('category')
movie_map=df['movieId'].astype('category')

In [ ]:
row=user_map.cat.codes
col=movie_map.cat.codes
data=df['rating'].values

In [ ]:
coo=coo_matrix((data,(row,col)))

In [ ]:
csr=coo.tocsr()

In [ ]:
from sklearn.neighbors import NearestNeighbors
model=NearestNeighbors(metric='cosine',algorithm='brute')
model.fit(csr.T)

NearestNeighbors(algorithm='brute', metric='cosine')

In [ ]:
df[df['title'].str.contains('Iron Man')]

,movieId,title,genres,userId,rating,timestamp
87149,59315,Iron Man,Action|Adventure|Sci-Fi,15,2.0,1510572096
87150,59315,Iron Man,Action|Adventure|Sci-Fi,18,4.0,1455618310
87151,59315,Iron Man,Action|Adventure|Sci-Fi,21,4.0,1418063513
87152,59315,Iron Man,Action|Adventure|Sci-Fi,22,0.5,1268727313
87153,59315,Iron Man,Action|Adventure|Sci-Fi,25,4.0,1535470521
...,...,...,...,...,...,...
95581,102125,Iron Man 3,Action|Sci-Fi|Thriller|IMAX,596,4.0,1535709834
95582,102125,Iron Man 3,Action|Sci-Fi|Thriller|IMAX,599,3.0,1498523185
95583,102125,Iron Man 3,Action|Sci-Fi|Thriller|IMAX,610,5.0,1479545231
99365,142056,Iron Man & Hulk: Heroes United,Action|Adventure|Animation,550,3.0,1488728343


In [ ]:
movie_map

,movieId
0,1
1,1
2,1
3,1
4,1
...,...
100831,193581
100832,193583
100833,193585
100834,193587


In [ ]:
movie_index = movie_map.cat.categories.get_loc(59315)

In [ ]:
movie_index

6726

In [ ]:
csr.shape

(610, 9724)

In [ ]:
#movie_index=59315

In [ ]:
distance,suggestion=model.kneighbors(csr.T[movie_index],n_neighbors=10)

In [ ]:
distance

array([[0.        , 0.32946219, 0.33222512, 0.34132449, 0.35801447,
        0.37573223, 0.40292874, 0.40326435, 0.41050844, 0.41102912]])

In [ ]:
suggestion#0 index is same input because it very near

array([[6726, 6693, 6755, 7675, 7307, 7195, 5901, 7022, 7571, 8457]])

In [ ]:
for i in range(len(suggestion)):
  print(df['title'][suggestion[i]])

6726    Interview with the Vampire: The Vampire Chroni...
6693    Interview with the Vampire: The Vampire Chroni...
6755    Interview with the Vampire: The Vampire Chroni...
7675                                             Outbreak
7307                          Madness of King George, The
7195    Like Water for Chocolate (Como agua para choco...
5901                                        Billy Madison
7022                   Star Wars: Episode IV - A New Hope
7571                                 Natural Born Killers
8457                                             Stargate
Name: title, dtype: object


In [ ]:
titles=df['title'].iloc[suggestion.flatten()].drop_duplicates()
for i in titles:
  print(i)

Interview with the Vampire: The Vampire Chronicles
Outbreak
Madness of King George, The
Like Water for Chocolate (Como agua para chocolate)
Billy Madison
Star Wars: Episode IV - A New Hope
Natural Born Killers
Stargate
